<a href="https://colab.research.google.com/github/ai-agent-yonsei/agent/blob/main/7_1_Multi_Agent_Framework_II_Chapter1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Chapter 1. Orchestration & Routing
  


| 실습 | 주제 |  
|---|---|
| 0 | 공통 설정 (MOCK/REAL, 시크릿, 호출 기록) |  
| 1-1 | 오케스트레이터의 세 결정: 정적 vs 동적 |  
| 1-2 | superstep·reducer·barrier |  
| 1-3 | 다섯 가지 제어 도구 |  
| 1-4 | 동적 감독자와 겹겹이 종료 장치 |
| 1-5 | 라우터 3종과 정책 |  
| 1-6 | cascade 라우팅 |  
| 1-7 | 다중 의도 fan-out/fan-in |  
| 1-8 | 오분류 비용 행렬 |  

---
## 실습 0. 공통 설정 (패키지·모드 스위치·시크릿·호출 기록)


- **목표**: MOCK/REAL 전환과 호출 기록 장치를 준비함

In [ ]:
# ===== 공통 설정 (실습 0) =====
# 필요한 패키지를 설치함. 코랩에는 대부분 이미 있어서 금방 끝남
%pip install -q langgraph openai scikit-learn pandas

import os, re, json, time, operator, random
from typing import Annotated, TypedDict, Literal
import pandas as pd

# ---- 실행 모드 스위치 ----
USE_REAL_LLM = False      # False=MOCK(모의 응답, 비용 0), True=OpenAI API 실제 호출임
MODEL = "gpt-4o-mini"     # 실제 호출 시 쓸 모델명임. 계정에서 쓸 수 있는 모델로 바꿔도 됨

# ---- OpenAI 클라이언트 (코랩 시크릿 사용) ----
client = None
if USE_REAL_LLM:
    from google.colab import userdata          # 코랩 시크릿을 읽는 모듈임
    from openai import OpenAI
    # 왼쪽 열쇠(시크릿) 메뉴에 OPENAI_API_KEY를 등록하고 노트북 액세스를 켜야 함
    client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

# ---- 호출 기록 ----
CALL_LOG = []             # 모든 LLM 호출을 여기에 쌓음

def est_tokens(text):
    # MOCK용 토큰 어림 계산임. 글자 수의 절반으로 가정함 (대략적인 값임)
    return max(1, round(len(text) / 2))

def call_llm(system, user, mock, tag="llm", json_mode=False):
    """LLM 호출 공통 함수임. MOCK이면 mock 값을, REAL이면 OpenAI 응답을 반환함"""
    t0 = time.time()
    if USE_REAL_LLM:
        # json_mode면 JSON 객체로만 답하게 강제함 (프롬프트에 'JSON' 단어가 있어야 함)
        kwargs = {"response_format": {"type": "json_object"}} if json_mode else {}
        resp = client.chat.completions.create(
            model=MODEL, temperature=0,
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": user}], **kwargs)
        text = resp.choices[0].message.content
        in_tok, out_tok = resp.usage.prompt_tokens, resp.usage.completion_tokens
    else:
        # MOCK: API를 부르지 않고 미리 정한 응답을 씀. 호출은 똑같이 기록함
        text = mock() if callable(mock) else mock
        in_tok, out_tok = est_tokens(system + user), est_tokens(text)
    CALL_LOG.append({"tag": tag, "in_tok": in_tok, "out_tok": out_tok,
                     "sec": round(time.time() - t0, 3)})
    return text

def reset_log():
    CALL_LOG.clear()          # 실험마다 기록을 비우고 시작함

def log_summary():
    # 호출 수와 토큰 합계를 돌려줌
    return {"calls": len(CALL_LOG),
            "in_tok": sum(c["in_tok"] for c in CALL_LOG),
            "out_tok": sum(c["out_tok"] for c in CALL_LOG)}

print("모드:", "REAL (OpenAI 호출)" if USE_REAL_LLM else "MOCK (모의 응답)")

### 코드별 필요성과 의미

| 코드 | 왜 필요한가 | 어떤 의미인가 |
|---|---|---|
| `USE_REAL_LLM` 스위치 | 비용 없이 먼저 흐름을 익히고, 필요할 때만 실제 API를 쓰려고 둠 | MOCK은 API를 부르지 않고 미리 정한 응답으로 호출 횟수만 똑같이 세는 방식임 |
| `userdata.get(...)` | API 키를 코드에 직접 쓰면 유출 위험이 있어서 코랩 시크릿에서 읽음 | 키가 노트북 파일에 남지 않음 |
| `call_llm()` | 모든 LLM 호출을 한 함수로 모아 호출 수·토큰을 한곳에서 셈 | Chapter 3의 비용·관측 논의가 이 기록 위에서 성립함 |
| `est_tokens()` | MOCK에는 실제 토큰 수가 없어서 어림값이 필요함 | 절대값보다 줄어드는 방향과 비율을 보는 용도임 |


---
## 실습 1-1. 오케스트레이터의 세 결정: 정적 vs 동적


- **목표**: 다음 실행자·넘길 정보·종료를 한 루프 안에서 확인하고, 정적/LLM 방식의 차이를 비교함

In [ ]:
# 담당 에이전트 3개를 단순 함수로 만듦. LLM 없이 '흐름'만 보려는 목적임
def agent_account(ctx): return "[account] 잔액·거래 조회 안내"
def agent_product(ctx): return "[product] 상품·금리 안내"
def agent_care(ctx):    return "[care] 민원 접수 안내"
AGENTS = {"account": agent_account, "product": agent_product, "care": agent_care}

# 의도별 단어 사전임. MOCK에서 '정답 역할'을 하고, 아래 실습에서도 다시 씀
INTENT_WORDS = {
    "account": ["잔액", "이체", "거래 내역", "비밀번호", "통장", "계좌"],
    "product": ["금리", "가입", "상품", "대출", "적금", "예금", "펀드", "연금"],
    "care":    ["항의", "민원", "불친절", "피해", "환불", "부당", "해명"],
}

def keyword_labels(text):
    # 텍스트에 단어가 하나라도 있는 의도를 모두 돌려줌
    return [k for k, ws in INTENT_WORDS.items() if any(w in text for w in ws)]

In [ ]:
# 오케스트레이터 루프임. 세 가지 결정이 코드에 그대로 드러남
def orchestrate(request, decide_next, max_steps=5):
    state = {"request": request, "done": [], "answers": {}}
    for _ in range(max_steps):                  # ③ 종료: 단계 상한임
        nxt = decide_next(state)                # ① 제어 흐름: 다음에 누가 실행되는가
        if nxt == "FINISH":                     # ③ 종료: 목표 달성임
            break
        ctx = {"request": request}              # ② 정보 흐름: 필요한 것만 넘김
        state["answers"][nxt] = AGENTS[nxt](ctx)
        state["done"].append(nxt)
    return state

In [ ]:
# [정적] 개발자가 경로를 고정함. 요청이 뭐든 account → product 순서로 실행함
def static_next(state):
    plan = ["account", "product"]
    left = [a for a in plan if a not in state["done"]]
    return left[0] if left else "FINISH"

# [동적] LLM이 요청과 진행 상황을 보고 다음 담당을 고름
def llm_next(state):
    needed = keyword_labels(state["request"])               # MOCK의 정답 역할임
    left = [a for a in needed if a not in state["done"]]
    mock_answer = {"next": left[0] if left else "FINISH"}   # MOCK이 돌려줄 응답임
    out = call_llm(
        system='고객센터 감독자임. 요청과 완료 목록을 보고 다음 담당(account/product/care) 또는 FINISH를 JSON {"next": ...}으로만 답함',
        user=json.dumps({"request": state["request"], "done": state["done"]}, ensure_ascii=False),
        mock=json.dumps(mock_answer), tag="supervisor", json_mode=True)
    return json.loads(out)["next"]

In [ ]:
tests = [
    "계좌 잔액 확인하고 남는 돈으로 가입할 예금 금리도 알려주세요",  # 두 의도임
    "서비스가 불친절해서 민원을 넣고 싶어요",                         # care임
    "적금 금리가 궁금해요",                                          # product임
]
rows = []
for req in tests:
    for name, fn in [("정적", static_next), ("LLM 동적", llm_next)]:
        reset_log()                              # 비교마다 호출 기록을 비움
        st = orchestrate(req, fn)
        rows.append({"요청": req[:18] + "…", "방식": name,
                     "실행된 에이전트": st["done"], "LLM 호출 수": log_summary()["calls"]})
pd.DataFrame(rows)

### 코드별 필요성과 의미

| 코드 | 왜 필요한가 | 어떤 의미인가 |
|---|---|---|
| `orchestrate()` | 세 결정(다음 실행자·넘길 정보·종료)이 한 곳에 보이게 하려고 만듦 | 오케스트레이터는 일을 하지 않고 흐름만 정한다는 슬라이드 5의 구조임 |
| `static_next()` | 코드가 경로를 정하는 방식의 기준선이 필요함 | 예측 가능하고 LLM 호출 0회지만 요청이 바뀌어도 같은 경로를 탐 |
| `llm_next()` | LLM이 경로를 정하는 방식을 같은 틀에서 비교하려고 만듦 | 유연하지만 매 단계 LLM 호출이 생김. MOCK에서는 사전이 정답 역할을 함 |
| `max_steps` | LLM이 FINISH를 안 내도 끝나게 하려고 둠 | 종료 제어의 가장 단순한 상한임 (실습 1-4에서 확장함) |

 정적 방식은 care 요청에도 account·product를 실행하고 정작 care는 놓침. 동적 방식은 필요한 담당만 실행하지만 호출 수가 늘어남

---
## 실습 1-2. superstep·reducer·barrier (LangGraph 실행 모델)


- **목표**: 병렬 노드가 한 단계에서 동시에 돌고, reducer로 합쳐지고, 모두 끝나야 다음으로 넘어가는 것을 확인함

In [ ]:
from langgraph.graph import StateGraph, START, END

# 상태 정의임. notes에 reducer(operator.add)를 붙여 병렬 결과를 이어 붙임
class S1(TypedDict):
    request: str
    notes: Annotated[list, operator.add]
    timeline: Annotated[list, operator.add]

T0 = time.time()
def stamp():
    return round(time.time() - T0, 2)           # 시작 후 경과 시간(초)임

def mk_worker(name, delay):
    # 워커 노드를 만들어 줌. delay만큼 일하는 척 쉼
    def node(state):
        time.sleep(delay)
        # 전체 상태가 아니라 바꿀 키만 반환함
        return {"notes": [f"{name} 결론"], "timeline": [f"{stamp()}s {name} 완료"]}
    return node

def join(state):
    # 병렬 노드가 모두 끝난 뒤에만 실행됨
    return {"timeline": [f"{stamp()}s join 실행, notes={state['notes']}"]}

g = StateGraph(S1)
g.add_node("account", mk_worker("account", 0.3))
g.add_node("product", mk_worker("product", 0.1))
g.add_node("care",    mk_worker("care", 0.2))
g.add_node("join", join)
for n in ["account", "product", "care"]:
    g.add_edge(START, n)                         # 세 노드가 같은 단계에서 병렬 실행됨
g.add_edge(["account", "product", "care"], "join")  # 배리어: 셋 다 끝나야 join 실행함
g.add_edge("join", END)
app = g.compile()

T0 = time.time()
out = app.invoke({"request": "테스트", "notes": [], "timeline": []})
print("\n".join(out["timeline"]))
print("notes:", out["notes"])

In [ ]:
# reducer가 없으면 어떻게 되는지 봄
class S1bad(TypedDict):
    notes: list                                   # reducer 없는 일반 키임

def w1(state): return {"notes": ["a"]}
def w2(state): return {"notes": ["b"]}

g2 = StateGraph(S1bad)
g2.add_node("w1", w1); g2.add_node("w2", w2)
g2.add_edge(START, "w1"); g2.add_edge(START, "w2")
g2.add_edge("w1", END);   g2.add_edge("w2", END)
try:
    g2.compile().invoke({"notes": []})
except Exception as e:
    print(type(e).__name__, "→", str(e).splitlines()[0])   # 병렬 쓰기 충돌 오류임

In [ ]:
# 그래프 구조를 mermaid 텍스트로 출력함 (mermaid.live 에 붙여 넣으면 그림으로 보임)
print(app.get_graph().draw_mermaid())

### 코드별 필요성과 의미

| 코드 | 왜 필요한가 | 어떤 의미인가 |
|---|---|---|
| `Annotated[list, operator.add]` | 병렬 노드가 같은 키에 쓰는 순간 합치는 규칙이 필요함 | 키마다 reducer를 붙이는 것이 공유 상태 설계의 출발점임 |
| `return {"notes": [...]}` | 노드가 상태 전체를 덮어쓰면 다른 노드의 결과가 사라짐 | 노드는 자기가 바꿀 키만 반환함 (슬라이드 7의 2단계) |
| `add_edge([...], "join")` | 병렬 결과를 모아 쓰는 노드는 모두 끝난 뒤에 실행돼야 함 | 배리어임. 가장 느린 노드(0.3초)가 그 단계의 지연을 정함 |
| reducer 없는 그래프 | reducer가 왜 필요한지 오류로 직접 확인하려고 둠 | 병렬 쓰기 충돌은 조용히 넘어가지 않고 오류가 남 |

 timeline에서 세 워커는 서로 기다리지 않고 거의 동시에 시작함. join은 가장 느린 0.3초 뒤에 실행됨. notes에는 세 결론이 모두 남음

---
## 실습 1-3. 다섯 가지 제어 도구: 정적 엣지·조건부 엣지·Command·Send·서브그래프

- **목표**: 같은 문제를 다섯 도구로 각각 구현해 통제력과 유연성의 차이를 확인함

In [ ]:
from langgraph.types import Command, Send

class P(TypedDict):
    text: str
    label: str
    log: Annotated[list, operator.add]
    results: Annotated[list, operator.add]

# ---- (a) 정적 엣지: 항상 A → B ----
def step_a(s): return {"log": ["A"]}
def step_b(s): return {"log": ["B"]}
g = StateGraph(P)
g.add_node("A", step_a); g.add_node("B", step_b)
g.add_edge(START, "A"); g.add_edge("A", "B"); g.add_edge("B", END)
print("(a) 정적 엣지   :", g.compile().invoke({"text": "x", "log": [], "results": []})["log"])

In [ ]:
# ---- (b) 조건부 엣지: 상태를 보고 함수가 다음 노드를 고름 ----
def classify(s):
    labels = keyword_labels(s["text"]) or ["care"]
    return {"label": labels[0], "log": ["classify"]}
def n_account(s): return {"log": ["account"]}
def n_product(s): return {"log": ["product"]}
def n_care(s):    return {"log": ["care"]}
def route_fn(s):  return s["label"]               # 반환값이 다음 노드 이름임

g = StateGraph(P)
for name, fn in [("classify", classify), ("account", n_account), ("product", n_product), ("care", n_care)]:
    g.add_node(name, fn)
g.add_edge(START, "classify")
g.add_conditional_edges("classify", route_fn, {"account": "account", "product": "product", "care": "care"})
for n in ["account", "product", "care"]:
    g.add_edge(n, END)
print("(b) 조건부 엣지:", g.compile().invoke({"text": "적금 금리", "log": [], "results": []})["log"])

In [ ]:
# ---- (c) Command: 이동(goto)과 상태 갱신(update)을 한 번에 반환 ----
def triage(s) -> Command[Literal["account", "product"]]:
    labels = keyword_labels(s["text"])
    dest = "product" if "product" in labels else "account"
    return Command(goto=dest, update={"label": dest, "log": ["triage→" + dest]})

g = StateGraph(P)
g.add_node("triage", triage); g.add_node("account", n_account); g.add_node("product", n_product)
g.add_edge(START, "triage")                        # triage 뒤 엣지는 없음. goto로 이동함
g.add_edge("account", END); g.add_edge("product", END)
print("(c) Command     :", g.compile().invoke({"text": "적금 금리", "log": [], "results": []})["log"])

In [ ]:
# ---- (d) Send: 실행 시점에 병렬 작업 수를 정함 (map-reduce) ----
def planner(s): return {"log": ["planner"]}
def fan_out(s):
    # 의도 개수만큼 worker를 동적으로 만듦. 의도가 3개면 3개가 뜸
    return [Send("worker", {"text": s["text"], "label": l}) for l in keyword_labels(s["text"])]
def worker(s): return {"results": [s["label"] + " 답변"]}

g = StateGraph(P)
g.add_node("planner", planner); g.add_node("worker", worker)
g.add_edge(START, "planner")
g.add_conditional_edges("planner", fan_out, ["worker"])
g.add_edge("worker", END)
r = g.compile().invoke({"text": "잔액 확인하고 예금 금리 알려줘", "log": [], "results": []})
print("(d) Send        :", sorted(r["results"]))

In [ ]:
# ---- (e) 서브그래프: 팀 내부 그래프가 자기 상태를 가지고 스스로 결정 ----
class Inner(TypedDict):
    text: str
    scratch: str                                   # 내부 전용 키임. 바깥 상태에 없음
    results: Annotated[list, operator.add]

def inner_a(s): return {"scratch": "내부 메모", "results": ["inner 결론"]}
ig = StateGraph(Inner)
ig.add_node("inner_a", inner_a)
ig.add_edge(START, "inner_a"); ig.add_edge("inner_a", END)
inner_app = ig.compile()

g = StateGraph(P)
g.add_node("team", inner_app)                      # 컴파일된 그래프를 노드로 끼움
g.add_edge(START, "team"); g.add_edge("team", END)
r = g.compile().invoke({"text": "x", "log": [], "results": []})
print("(e) 서브그래프  :", r["results"], "| scratch 노출 여부:", "scratch" in r)

### 코드별 필요성과 의미

| 코드 | 왜 필요한가 | 어떤 의미인가 |
|---|---|---|
| 정적 엣지 (a) | 가장 예측 가능한 기준선이 필요함 | 항상 같은 순서임. 테스트·감사가 가장 쉬움 |
| 조건부 엣지 (b) | 상태에 따라 갈라지는 규칙 분기가 필요함 | 분기 규칙이 코드(route_fn)에 있어 설명 가능함 |
| `Command` (c) | 분기 결정과 상태 갱신을 한 노드가 같이 해야 할 때 씀 | 감독자·핸드오프가 쓰는 도구임. 엣지가 아니라 goto로 이동함 |
| `Send` (d) | 처리할 일의 개수가 실행해 봐야 정해질 때 씀 | fan-out의 기본 도구임. 슬라이드 13의 다중 의도 처리에 씀 |
| 서브그래프 (e) | 팀 내부 상태를 바깥에 노출하지 않으려고 씀 | 캡슐화와 상태 격리임. scratch가 바깥에 안 나옴 |

**관찰 포인트**: (a)→(e)로 갈수록 유연해지지만 경로를 코드만 보고 예측하기는 어려워짐. 필요한 만큼만 아래 단계로 내려가는 것이 실무 기본값임

---
## 실습 1-4. 동적 감독자와 겹겹이 종료 장치

- **목표**: 감독자 루프를 만들고 목표·상한·감시·안전망 네 겹 종료 장치를 하나씩 켜서 효과를 비교함

In [ ]:
from langgraph.errors import GraphRecursionError

class SV(TypedDict):
    request: str
    done: Annotated[list, operator.add]
    answers: Annotated[dict, lambda a, b: {**a, **b}]   # dict 병합 reducer임
    rounds: int
    final: str
    reason: str

# 종료 장치를 켜고 끄는 설정임
CFG = {"policy": "normal", "max_rounds": None, "progress_guard": False}

def choose_next(state):
    if CFG["policy"] == "buggy":
        # 버그 재현: LLM이 항상 같은 워커만 고름. 호출 수가 세어지도록 call_llm을 거침
        out = call_llm(system='JSON {"next": ...}으로만 답함', user=state["request"],
                       mock=json.dumps({"next": "account"}), tag="supervisor", json_mode=True)
        return json.loads(out)["next"]
    return llm_next({"request": state["request"], "done": state["done"]})  # 실습 1-1의 LLM 감독자 재사용함

def supervisor(state: SV) -> Command[Literal["account", "product", "care", "synth"]]:
    rounds = state.get("rounds", 0) + 1
    nxt = choose_next(state)
    # ② 상한: 라운드 상한을 넘으면 지금까지의 답으로 종료함
    if CFG["max_rounds"] and rounds > CFG["max_rounds"]:
        return Command(goto="synth", update={"rounds": rounds, "reason": "상한(max_rounds) 도달"})
    # ③ 감시: 이미 한 워커를 또 고르면 진전 없음으로 보고 개입 후 종료함
    if CFG["progress_guard"] and nxt in state.get("done", []):
        return Command(goto="synth", update={"rounds": rounds, "reason": "진전 없음 감지"})
    # ① 목표: 필요한 일이 끝났으면 정상 종료함
    if nxt == "FINISH":
        return Command(goto="synth", update={"rounds": rounds, "reason": "목표 달성"})
    return Command(goto=nxt, update={"rounds": rounds})

def mk_sv_worker(name):
    def node(state) -> Command[Literal["supervisor"]]:
        # 일을 마치면 supervisor로 돌아가 다음 경로를 다시 판단받음
        return Command(goto="supervisor", update={"done": [name], "answers": {name: AGENTS[name]({})}})
    return node

def synth(state):
    return {"final": " / ".join(state["answers"].values()) or "(결과 없음)"}

g = StateGraph(SV)
g.add_node("supervisor", supervisor)
for n in ["account", "product", "care"]:
    g.add_node(n, mk_sv_worker(n))
g.add_node("synth", synth)
g.add_edge(START, "supervisor"); g.add_edge("synth", END)
sv_app = g.compile()

In [ ]:
def run_case(title, cfg, req, limit=25):
    CFG.update(cfg)
    reset_log()
    try:
        out = sv_app.invoke({"request": req, "done": [], "answers": {}, "rounds": 0},
                            {"recursion_limit": limit})
        print(f"[{title}] 종료 사유={out['reason']} | 라운드={out['rounds']} | 감독 호출={log_summary()['calls']} | 결과={out['final'][:40]}")
    except GraphRecursionError:
        print(f"[{title}] ④ 안전망(recursion_limit={limit}) 발동 → 예외, 결과 없음")

Q7 = "계좌 잔액 확인하고 남는 돈으로 가입할 예금 금리도 알려주세요"
run_case("A 정상 정책        ", {"policy": "normal", "max_rounds": None, "progress_guard": False}, Q7)
run_case("B 버그+장치 없음   ", {"policy": "buggy",  "max_rounds": None, "progress_guard": False}, Q7, limit=12)
run_case("C 버그+상한       ", {"policy": "buggy",  "max_rounds": 4,    "progress_guard": False}, Q7)
run_case("D 버그+진전 감시  ", {"policy": "buggy",  "max_rounds": None, "progress_guard": True},  Q7)

### 코드별 필요성과 의미

| 코드 | 왜 필요한가 | 어떤 의미인가 |
|---|---|---|
| `supervisor()` + `Command` | 매 라운드 다음 경로를 다시 판단하는 감독자 루프가 필요함 | 워커가 끝나면 supervisor로 돌아오는 동적 방식임. 호출이 라운드마다 늘어남 |
| `CFG["max_rounds"]` | LLM이 FINISH를 못 내는 경우를 막으려고 둠 | ② 상한임. 지금까지 쌓인 답을 반환해서 쓸모 있는 결과가 남음 |
| `progress_guard` | 같은 워커를 반복 선택하는 헛돌기를 잡으려고 둠 | ③ 감시임. 개입 후 종료해서 낭비를 일찍 끊음 |
| `recursion_limit` | 앞 장치가 모두 뚫려도 무한 루프는 막아야 함 | ④ 안전망임. 예외가 나고 결과가 없으므로 마지막 수단이어야 함 |

 B는 예외만 남고 결과가 없음. C·D는 같은 버그에서도 현재 답을 남기고 끝남. 앞쪽 장치일수록 사용자에게 쓸모 있는 결과를 남김

---
## 실습 1-5. 라우터 3종(규칙·분류기·LLM)과 정책

- **슬라이드 매핑**: 슬라이드 10 (라우팅 = 분류 + 정책), 11 (라우터 3종)
- **목표**: 같은 18개 요청을 세 방식으로 분류해 보고, 애매하거나 위험한 경우를 받아내는 정책을 붙임

In [ ]:
# Y-Bank 가상 데이터임 (이 강의용으로 새로 만든 18건이라 슬라이드의 수치와 다를 수 있음)
# 형식: (요청, 정답 라벨 목록). 라벨이 2개면 다중 의도임
TEST = [
    ("계좌 잔액이 얼마인지 알려주세요", ["account"]),
    ("이번 달 이체 한도를 확인하고 싶어요", ["account"]),
    ("체크카드 재발급은 어떻게 하나요", ["account"]),
    ("예금 금리가 얼마인가요", ["product"]),
    ("적금 상품 가입 조건을 알려주세요", ["product"]),
    ("대출 한도와 금리를 비교하고 싶어요", ["product"]),
    ("계좌 잔액 확인하고 남는 돈으로 가입할 예금 금리도 알려주세요", ["account", "product"]),
    ("직원 응대가 너무 불친절했어요 항의합니다", ["care"]),
    ("보이스피싱 피해를 당한 것 같아요 도와주세요", ["care"]),
    ("수수료가 부당하게 빠져나갔어요 환불해 주세요", ["care"]),
    ("통장 거래 내역을 조회해 주세요", ["account"]),
    ("펀드 상품 추천해 주세요", ["product"]),
    ("비밀번호를 잊어버렸어요", ["account"]),
    ("연금저축 상품은 세제 혜택이 어떻게 되나요", ["product"]),
    ("민원을 접수하고 싶습니다", ["care"]),
    ("요즘 돈 관련해서 궁금한 게 있어요", ["product"]),
    ("카드값이 이상하게 나왔는데 해명해 주세요", ["care"]),
    ("적금 만기됐는데 계좌로 들어왔는지 확인하고 해지 방법도 알려주세요", ["account", "product"]),
]
GOLD = dict(TEST)                       # 요청 → 정답 조회용임. MOCK LLM이 이 표를 베낌

# 분류기 학습용 데이터임. 테스트 문장과 겹치지 않게 따로 만듦
TRAIN = [
    ("잔액 조회하고 싶어요", "account"), ("계좌 이체 방법 알려주세요", "account"),
    ("통장 입출금 내역 보여주세요", "account"), ("비밀번호 재설정은 어떻게 하나요", "account"),
    ("체크카드를 분실했어요 재발급 해주세요", "account"), ("자동이체 등록하고 싶어요", "account"),
    ("입금 내역 확인해 주세요", "account"), ("송금 한도를 올리고 싶어요", "account"),
    ("인터넷뱅킹 로그인이 안 돼요", "account"), ("잔고 확인 부탁해요", "account"),
    ("정기예금 금리 알려주세요", "product"), ("적금 가입하려고 해요", "product"),
    ("대출 상품 조건이 궁금해요", "product"), ("펀드 수익률은 어떤가요", "product"),
    ("연금 상품 추천해 주세요", "product"), ("주택담보대출 금리 비교해 주세요", "product"),
    ("보험 상품 가입 조건 알려주세요", "product"), ("ETF 투자 상품이 궁금해요", "product"),
    ("금리 우대 조건이 뭔가요", "product"), ("청약 상품 가입하고 싶어요", "product"),
    ("상담원이 불친절했어요", "care"), ("민원을 제기하고 싶습니다", "care"),
    ("수수료가 잘못 부과됐어요 취소해 주세요", "care"), ("사기를 당했어요 도와주세요", "care"),
    ("서비스에 항의하고 싶어요", "care"), ("부당한 요금 청구에 대해 해명 바랍니다", "care"),
    ("피해 보상은 어떻게 받나요", "care"), ("개인정보가 유출된 것 같아요", "care"),
    ("직원의 태도가 너무 불쾌했어요", "care"), ("고객센터 응대가 엉망이에요", "care"),
]

In [ ]:
# ---- 라우터 1: 규칙 ----
# 확실한 단어만 넣은 좁은 사전임. '정확히 한 라벨만' 맞으면 확정하고 아니면 통과시킴
RULES = {"account": ["잔액", "이체"], "product": ["금리"], "care": ["항의", "민원"]}

def rule_route(text):
    hit = [k for k, ws in RULES.items() if any(w in text for w in ws)]
    return hit if len(hit) == 1 else None          # 0개 또는 2개 이상이면 None(통과)임

# ---- 라우터 2: 분류기 ----
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

clf = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 3)),   # 글자 조각 기반이라 띄어쓰기 변화에 덜 민감함
    LogisticRegression(C=10, max_iter=1000))
clf.fit([t for t, _ in TRAIN], [y for _, y in TRAIN])

def clf_route(text):
    p = clf.predict_proba([text])[0]
    order = sorted(zip(clf.classes_, p), key=lambda x: -x[1])
    margin = order[0][1] - order[1][1]              # 1위와 2위 확률 차이임. 클수록 확신함
    return order[0][0], round(float(margin), 2), dict(zip(clf.classes_, p))

# ---- 라우터 3: LLM ----
def llm_route(text):
    mock_answer = {"labels": GOLD[text]}            # MOCK은 정답표를 그대로 돌려줌 (정답률 100%가 당연함)
    out = call_llm(
        system='고객센터 라우터임. 요청의 의도를 account/product/care 중 모두 골라 JSON {"labels": [...]}으로만 답함',
        user=text, mock=json.dumps(mock_answer), tag="router", json_mode=True)
    return json.loads(out)["labels"]

In [ ]:
# ---- 정책: 분류 결과를 받아 '어떻게 처리할지' 정함 ----
HIGH_RISK = ["보이스피싱", "사기", "유출"]

def apply_policy(text, labels, margin=None):
    if any(w in text for w in HIGH_RISK):
        return "human"                              # 고위험 → 사람에게
    if not labels:
        return "default_queue"                      # 매칭 없음 → 기본 대기열
    if margin is not None and margin < 0.1:
        return "clarify"                            # 확신 낮음 → 되묻기
    if len(labels) > 1:
        return "fan_out"                            # 다중 의도 → 분해해서 병렬 처리
    return "route:" + labels[0]                     # 그 외 → 해당 담당에게

rows = []
for text, gold in TEST:
    r = rule_route(text)
    c, m, _ = clf_route(text)
    llm_labels = llm_route(text)
    # 의도는 LLM 라벨을 쓰고, 확신도 신호는 분류기 margin을 씀 (규칙이 확정한 건은 신호가 필요 없어 None임)
    rows.append({"요청": text[:16] + "…", "정답": "+".join(gold),
                 "규칙": "+".join(r) if r else "통과",
                 "분류기": f"{c}({m})",
                 "LLM": "+".join(llm_labels),
                 "정책": apply_policy(text, llm_labels, None if r else m)})
df15 = pd.DataFrame(rows)
df15

In [ ]:
# 라우터별 성적을 요약함 (정확도는 정답 라벨 집합이 같을 때만 맞다고 봄)
rule_out = [rule_route(t) for t, _ in TEST]
covered = [(o, g) for o, (_, g) in zip(rule_out, TEST) if o]
clf_out = [[clf_route(t)[0]] for t, _ in TEST]
print(f"규칙   : 확정 {len(covered)}/{len(TEST)}건, 확정한 것의 정확도 {sum(set(o)==set(g) for o,g in covered)/len(covered):.2f}")
print(f"분류기 : 전체 정확도 {sum(set(o)==set(g) for o,(_,g) in zip(clf_out, TEST))/len(TEST):.2f} (다중 의도는 구조상 못 맞힘)")
print(f"LLM    : MOCK이라 정답표를 베낀 값이므로 정확도는 의미 없음. REAL로 바꿔야 실제 성능을 볼 수 있음")

### 코드별 필요성과 의미

| 코드 | 왜 필요한가 | 어떤 의미인가 |
|---|---|---|
| `TEST`, `GOLD` | 세 라우터를 같은 문제로 비교하려면 공통 정답표가 필요함 | MOCK LLM은 이 표를 베끼므로 LLM 정확도는 참고할 수 없음 |
| `rule_route()` | 싸고 설명 가능한 첫 단계가 필요함 | 정확히 한 라벨만 맞을 때만 확정함. 모호하면 통과시켜 다음 단계의 몫으로 남김 |
| `clf_route()`의 margin | 분류기가 얼마나 확신하는지 알아야 다음 단계로 넘길지 정할 수 있음 | 1위와 2위 확률 차이임. 작으면 경계 사례라는 신호임 |
| `llm_route()` | 표현이 달라도 의도를 읽고 다중 의도도 처리하는 단계가 필요함 | 가장 일반화가 좋지만 호출 비용과 비결정성이 있음 |
| `apply_policy()` | 분류만으로는 고위험·모호함·다중 의도를 안전하게 처리할 수 없음 | 좋은 라우터 = 분류기 + 경계 사례를 받아내는 정책임 |

 규칙은 12건을 통과시킴. 분류기는 다중 의도를 한 라벨로만 답해 구조상 못 맞히고, margin이 0에 가까운 건(6번 등)은 경계 사례 신호임. 정책은 보이스피싱을 human, 다중 의도를 fan_out, 확신 낮은 건을 clarify로 보냄

---
## 실습 1-6. cascade 라우팅: 싼 단계에서 먼저 끝내기

- **슬라이드 매핑**: 슬라이드 12 (싼 단계에서 먼저 끝내고 애매한 요청만 올림)
- **목표**: 규칙 → 분류기 → LLM 순으로 확신 있는 곳에서 끝내고, margin 값에 따른 LLM 호출 수와 정확도를 비교함

In [ ]:
def cascade(text, margin=0.3):
    # ① 규칙: 정확히 하나가 맞으면 확정함
    r = rule_route(text)
    if r:
        return r, "rule"
    # ② 분류기: 1·2위 확률 차이가 margin 이상이면 확정함
    label, m, _ = clf_route(text)
    if m >= margin:
        return [label], "classifier"
    # ③ LLM: 남은 애매한 요청만 판단함 (호출이 여기서만 생김)
    return llm_route(text), "llm"

def evaluate_cascade(margin):
    reset_log()
    stages, correct = [], 0
    for text, gold in TEST:
        pred, stage = cascade(text, margin)
        stages.append(stage)
        correct += set(pred) == set(gold)
    return {"margin": margin, "LLM 호출": log_summary()["calls"],
            "규칙": stages.count("rule"), "분류기": stages.count("classifier"), "LLM": stages.count("llm"),
            "정확도": round(correct / len(TEST), 2)}

# LLM만 쓰는 기준선임: 모든 요청이 LLM으로 감
reset_log()
llm_only_acc = sum(set(llm_route(t)) == set(g) for t, g in TEST) / len(TEST)
print(f"LLM 단독: 호출 {log_summary()['calls']}회, 정확도 {llm_only_acc:.2f}")

In [ ]:
# margin을 바꿔 가며 호출 수와 정확도의 균형을 봄
pd.DataFrame([evaluate_cascade(m) for m in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]])

### 코드별 필요성과 의미

| 코드 | 왜 필요한가 | 어떤 의미인가 |
|---|---|---|
| `cascade()`의 3단계 | 모든 요청을 비싼 LLM에 보내면 비용·지연이 커짐 | 확신이 있으면 거기서 끝내고 애매한 것만 올림 (슬라이드 12의 구조) |
| `margin` 임계값 | '확신이 있다'의 기준이 필요함 | 높이면 LLM 호출이 늘고 정확도가 오르며, 낮추면 반대임. 비용과 정확도의 교환 조건임 |
| LLM 단독 기준선 | 절감 효과는 비교 대상이 있어야 보임 | 호출 수를 몇 회에서 몇 회로 줄였는지가 핵심 지표임 |
| `evaluate_cascade()` | 임계값마다 같은 방식으로 재야 비교가 공정함 | 정확도·호출 수·단계별 처리 건수를 함께 봄 (슬라이드 14의 평가 지표) |

 margin이 낮을수록 LLM 호출은 줄지만 분류기가 틀릴 위험이 커짐. 정확도가 1.0에 도달하는 가장 낮은 margin이 적정 후보임 (이 데이터에서는 0.4 부근). 수치는 슬라이드(5/9/4건)와 달라도 정상임

---
## 실습 1-7. 다중 의도: 단일 라벨 vs 다중 라벨 + fan-out/fan-in

- **목표**: 한 문장 속 두 요청을 단일 라벨이 놓치는 것과, 다중 라벨 + Send로 모두 처리하는 것을 비교함

In [ ]:
# 상태 정의임. answers는 병렬 워커의 답을 이어 붙이는 reducer를 가짐
class MI(TypedDict):
    request: str
    labels: list
    answers: Annotated[list, operator.add]
    final: str

def router_multi(s):
    # 다중 라벨 라우팅임. 의도를 모두 뽑음
    return {"labels": llm_route(s["request"])}

def fan_out_mi(s):
    # 라벨 하나당 워커 하나를 병렬로 띄움 (fan-out)
    return [Send("worker", {"label": l, "request": s["request"]}) for l in s["labels"]]

def worker_mi(s):
    return {"answers": [AGENTS[s["label"]]({})]}

def synth_mi(s):
    # 모인 답을 하나로 합침 (fan-in)
    return {"final": " + ".join(sorted(s["answers"]))}

g = StateGraph(MI)
g.add_node("router", router_multi); g.add_node("worker", worker_mi); g.add_node("synth", synth_mi)
g.add_edge(START, "router")
g.add_conditional_edges("router", fan_out_mi, ["worker"])
g.add_edge("worker", "synth"); g.add_edge("synth", END)
multi_app = g.compile()

In [ ]:
def single_label_answer(text):
    # 단일 라벨 방식임: 첫 라벨 하나만 골라 그 담당에게만 보냄
    first = llm_route(text)[0]
    return AGENTS[first]({})

rows = []
for i in (6, 17, 3):                                         # 다중 의도 2건(7·18번째)과 비교용 단일 의도 1건임
    text, gold = TEST[i]
    single =single_label_answer(text)
    multi = multi_app.invoke({"request": text, "labels": [], "answers": []})["final"]
    covered_single = sum(l in single for l in gold)
    covered_multi = sum(l in multi for l in gold)
    rows.append({"요청": text[:20] + "…", "정답 라벨": "+".join(gold),
                 "단일 라벨 커버": f"{covered_single}/{len(gold)}",
                 "다중 라벨 커버": f"{covered_multi}/{len(gold)}"})
pd.DataFrame(rows)

### 코드별 필요성과 의미

| 코드 | 왜 필요한가 | 어떤 의미인가 |
|---|---|---|
| `router_multi()` | 의도가 둘이면 라벨도 여러 개 내야 함 | 분류 단계에서 다중 라벨을 허용하는 것이 출발점임 |
| `Send` 기반 `fan_out_mi()` | 의도 개수가 요청마다 달라서 워커 수를 미리 못 정함 | 실행 시점에 병렬 작업 수를 정하는 map 단계임 |
| `answers` reducer | 병렬 워커의 답이 서로 덮어쓰면 안 됨 | reduce 단계의 합치기 규칙임 |
| `synth_mi()` | 고객은 답 하나를 원함 | fan-in으로 합성하고, 합성 단계에서 두 답이 모두 들어갔는지 검증하는 것이 중요함 |

 단일 라벨은 다중 의도 문장의 절반만 커버함. 다중 라벨 + Send는 모든 정답 라벨을 커버함

---
## 실습 1-8. 오분류 비용 행렬과 비용 민감 라우팅

- **목표**: 정확도만이 아니라 오분류 비용으로 라우터를 평가하고, 비용을 줄이는 결정 규칙을 적용함

In [ ]:
# 오분류 비용 행렬임. 행=정답, 열=예측. care를 놓치면 비용 5, 일반을 care로 보내면 2임
LABELS = ["account", "product", "care"]
COST = {
    "account": {"account": 0, "product": 1, "care": 2},
    "product": {"account": 1, "product": 0, "care": 2},
    "care":    {"account": 5, "product": 5, "care": 0},
}

def case_cost(gold, pred):
    # 정답 라벨 중 예측에 없는 것마다, 예측 중 가장 싼 오류 비용을 더함. 예측이 비면 care 비용으로 계산함
    pred = pred or ["care"]
    return sum(min(COST[g][p] for p in pred) for g in gold if g not in pred)

def score(preds):
    acc = sum(set(p) == set(g) for p, (_, g) in zip(preds, TEST)) / len(TEST)
    cost = sum(case_cost(g, p) for p, (_, g) in zip(preds, TEST))
    return acc, cost

In [ ]:
# 전략별 예측을 만듦
reset_log()
llm_preds = [llm_route(t) for t, _ in TEST]
llm_calls = log_summary()["calls"]

rule_preds = [rule_route(t) or ["care"] for t, _ in TEST]        # 규칙만: 못 맞히면 안전하게 care로 보냄
clf_preds  = [[clf_route(t)[0]] for t, _ in TEST]                # 분류기만: 1위 라벨 그대로임

def cost_sensitive_pred(text):
    # 기대 비용이 가장 작은 라벨을 고름: 각 후보 p에 대해 sum(P(정답=t) * COST[t][p])임
    _, _, proba = clf_route(text)
    best = min(LABELS, key=lambda p: sum(proba[t] * COST[t][p] for t in LABELS))
    return [best]
cs_preds = [cost_sensitive_pred(t) for t, _ in TEST]

reset_log()
casc_preds = [cascade(t, 0.3)[0] for t, _ in TEST]
casc_calls = log_summary()["calls"]

result = []
for name, preds, calls in [("LLM 단독", llm_preds, llm_calls), ("규칙만(미매칭→care)", rule_preds, 0),
                           ("분류기만", clf_preds, 0), ("분류기+비용 민감 결정", cs_preds, 0),
                           ("cascade(margin 0.3)", casc_preds, casc_calls)]:
    acc, cost = score(preds)
    result.append({"전략": name, "정확도": round(acc, 2), "총 오분류 비용": cost, "LLM 호출": calls})
pd.DataFrame(result)

### 코드별 필요성과 의미

| 코드 | 왜 필요한가 | 어떤 의미인가 |
|---|---|---|
| `COST` 행렬 | 오류마다 업무 영향이 달라서 정확도 하나로는 부족함 | care를 놓치는 비용(5)이 과잉 전달 비용(2)보다 큼. 비용은 예시값이며 업무 정책으로 정함 |
| `case_cost()` | 다중 의도에서 일부만 놓친 경우도 비용으로 재야 함 | 놓친 정답 라벨마다 비용을 더함 |
| `cost_sensitive_pred()` | 확률이 비슷할 때 비싼 오류를 피하는 쪽으로 기울이려고 씀 | 기대 비용 최소화임. 정확도가 같아도 총 비용이 줄 수 있음 |
| 전략 비교 표 | 정확도·비용·호출 수를 한 표에서 봐야 선택할 수 있음 | 슬라이드 14의 평가 지표(정확도 + 오분류 비용 + LLM 호출 비율 등)를 실제로 계산한 것임 |

 정확도가 비슷한 전략끼리 총 오분류 비용이 다를 수 있음. 비용 민감 결정은 care 쪽으로 기울어 비싼 오류를 줄이는 경향이 있음